Song similarity: which songs are “close” to each other in audio-feature space and in tag space? Given a song, make a list of approximate nearest neighbours (similar songs) found via LSH, or other methods


In [3]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("../dataset_with_lastfm_all.csv")

In [7]:
feature_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "lfm_playcount_log", "lfm_listeners_log"
]

# 1) First create the log columns
df["lfm_playcount_log"] = np.log1p(df["lfm_playcount"])
df["lfm_listeners_log"] = np.log1p(df["lfm_listeners"])

# 2) Drop rows where any of the feature columns are NaN
df_lsh = df.dropna(subset=feature_cols).copy()

print(df.shape[0], "rows before")
print(df_lsh.shape[0], "rows after dropping NaNs")

# 3) Build X from the clean frame
X = df_lsh[feature_cols].to_numpy().astype(float)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.decomposition import PCA
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)


73608 rows before
72840 rows after dropping NaNs


In [8]:
import numpy as np
from collections import defaultdict

class RandomHyperplaneLSH:
    def __init__(self, n_features, n_tables=10, n_bits=16, random_state=None):
        """
        n_features: dimensionality d of your feature vectors
        n_tables (L): number of hash tables
        n_bits (k): number of hyperplanes per table
        """
        self.n_features = n_features
        self.n_tables = n_tables
        self.n_bits = n_bits
        self.random_state = np.random.RandomState(random_state)

        # Random hyperplanes: shape (L, k, d)
        self.hyperplanes = self.random_state.randn(n_tables, n_bits, n_features)

        # Hash tables: list of dicts: code (int) -> list of item indices
        self.tables = [defaultdict(list) for _ in range(n_tables)]

        # We’ll store the original vectors for re-ranking
        self.data = None

    def _hash_vector(self, x):
        """
        x: shape (d,)
        returns list of L integer hash codes
        """
        # (L, k, d) dot (d,) -> (L, k)
        projections = np.tensordot(self.hyperplanes, x, axes=[2, 0])
        # sign -> bits (0/1)
        bits = (projections >= 0).astype(int)
        # convert each row of bits to an integer code
        codes = []
        for l in range(self.n_tables):
            # interpret bits[l] as binary number
            code = 0
            for b in bits[l]:
                code = (code << 1) | int(b)
            codes.append(code)
        return codes

    def fit(self, X):
        """
        X: shape (n_samples, d)
        """
        n_samples, d = X.shape
        assert d == self.n_features
        self.data = X

        for idx, x in enumerate(X):
            codes = self._hash_vector(x)
            for l, code in enumerate(codes):
                self.tables[l][code].append(idx)

    def query(self, q, max_candidates=500, top_k=20):
        """
        q: query vector shape (d,)
        Returns top_k nearest neighbours (indices and distances).
        """
        from numpy.linalg import norm

        codes = self._hash_vector(q)
        candidate_idxs = set()
        for l, code in enumerate(codes):
            bucket = self.tables[l].get(code, [])
            candidate_idxs.update(bucket)

        candidate_idxs = list(candidate_idxs)
        if len(candidate_idxs) > max_candidates:
            candidate_idxs = candidate_idxs[:max_candidates]

        X_candidates = self.data[candidate_idxs]

        # cosine distances (1 - cosine similarity)
        norms_q = norm(q)
        norms_c = norm(X_candidates, axis=1)
        sims = (X_candidates @ q) / (norms_c * norms_q + 1e-10)
        dists = 1 - sims

        order = np.argsort(dists)[:top_k]
        return [(candidate_idxs[i], float(dists[i])) for i in order]



In [10]:
# Suppose X_pca is your feature matrix (n_tracks x d)
n_features = X_pca.shape[1]

lsh = RandomHyperplaneLSH(
    n_features=n_features,
    n_tables=10,
    n_bits=16,
    random_state=42
)
lsh.fit(X_pca)

# Function: given track_id, get similar songs
id_to_idx = {tid: i for i, tid in enumerate(df_lsh["track_id"].values)}
idx_to_id = {i: tid for tid, i in id_to_idx.items()}

def get_similar_tracks(track_id, top_k=20):
    idx = id_to_idx[track_id]
    q_vec = X_pca[idx]
    neighbours = lsh.query(q_vec, top_k=top_k+1)  # +1 to include self

    results = []
    for neigh_idx, dist in neighbours:
        if neigh_idx == idx:
            continue  # skip self
        row = df_lsh.iloc[neigh_idx]
        results.append({
            "track_id": row["track_id"],
            "track_name": row["track_name"],
            "artists": row["artists"],
            "distance": dist
        })
        if len(results) == top_k:
            break
    return results

# Example:
similar = get_similar_tracks(df_lsh.loc[0, "track_id"], top_k=10)
#print them
for item in similar:
    print(f"{item['track_name']} by {item['artists']} (distance: {item['distance']:.4f})")


i can't get high by Royal & the Serpent (distance: 0.0754)
It Wasn't Me by Shaggy;Ricardo Ducent (distance: 0.0853)
Our Day Will Come by Amy Winehouse (distance: 0.0859)
Ms. Fat Booty by Mos Def (distance: 0.0883)
What I Got by Sublime (distance: 0.0887)
Just a Friend by Biz Markie (distance: 0.0900)
Make Me Feel by Janelle Monáe (distance: 0.0933)
Runnin' by The Pharcyde (distance: 0.0943)
Who Knows by Protoje;Chronixx (distance: 0.0945)
Écoute Chérie by Vendredi sur Mer (distance: 0.0973)


In [26]:
#find the track id of "Shape of You" by Ed Sheera
shape_of_you_id = df_lsh[df_lsh["track_name"] == "Shape of You"]["track_id"].values[0]

#print similar songs to "Shape of You"
similar_to_shape_of_you = get_similar_tracks(shape_of_you_id, top_k=10)
for item in similar_to_shape_of_you:
    print(f"{item['track_name']} by {item['artists']} (distance: {item['distance']:.4f})") 
    
    

No Ceiling by Eddie Vedder (distance: 0.0313)
Vampire by Dominic Fike (distance: 0.0332)
Calm Down (with Selena Gomez) by Rema;Selena Gomez (distance: 0.0338)
Tick Tock (feat. 24kGoldn) by Clean Bandit;Mabel;24kGoldn (distance: 0.0343)
Hecha Pa' Mi by Boza (distance: 0.0347)
Get Busy by Sean Paul (distance: 0.0396)
Somebody! by Loco;Hwa Sa (distance: 0.0407)
Quiere Beber by Anuel AA (distance: 0.0418)
Brujeria by El Gran Combo De Puerto Rico (distance: 0.0424)
Colgando en tus manos (con Marta Sánchez) by Carlos Baute;Marta Sánchez (distance: 0.0424)
